# Step 8 — Selected figure generation

This notebook runs the six standalone figure scripts and displays their composite outputs. From the public repository it uses `figures/plot_figure1.R` through `plot_figure6.R`, `figures/data/`, and `figures/outputs/`. From the research project it uses the corresponding `8-Visualization/scripts/`, `8-Visualization/data/`, and `8-Visualization/figures/` directories. The setup locates either layout from the working directory or its parents.

R is selected from `ICCA_R_HOME`, then `R_HOME`, or `R RHOME` on PATH. Each figure cell reuses existing non-empty PDF and PNG composites. Set `force_rerun=True` in that cell to redraw from the staged tables, stored fits, and display coordinates.

Scientific identifiers remain C1/C2/C3. Patient denominators, cohort restrictions, and assay-specific interpretations follow the underlying source tables and figure legends. The fixed 17-gene development candidate retains its exploratory scope.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path
from IPython.display import Image, display


def locate_figure_layout(start):
    start = Path(start).resolve()
    for root in (start, *start.parents):
        if (root / "8-Visualization/scripts/plot_figure1.R").is_file():
            return (root, root / "8-Visualization/scripts",
                    root / "8-Visualization/data", root / "8-Visualization/figures")
        if (root / "figures/plot_figure1.R").is_file():
            return root, root / "figures", root / "figures/data", root / "figures/outputs"
    raise FileNotFoundError("Cannot locate figures/plot_figure1.R or 8-Visualization/scripts/plot_figure1.R")


def locate_r_runtime():
    configured = os.environ.get("ICCA_R_HOME") or os.environ.get("R_HOME")
    if configured:
        r_home = Path(configured).expanduser().resolve()
    else:
        command = shutil.which("R")
        if command is None:
            raise FileNotFoundError("Set ICCA_R_HOME or R_HOME, or make R available on PATH")
        result = subprocess.run([command, "RHOME"], check=True, capture_output=True, text=True)
        r_home = Path(result.stdout.strip()).expanduser().resolve()
    for relative_path in ("bin/exec/R", "bin/R", "bin/R.exe"):
        executable = r_home / relative_path
        if executable.is_file():
            environment = os.environ.copy()
            environment["R_HOME"] = str(r_home)
            return executable, environment
    raise FileNotFoundError(f"No R executable found under {r_home}")


ROOT, SCRIPT_DIR, DATA_ROOT, OUTPUT_ROOT = locate_figure_layout(Path.cwd())
R_EXEC, R_ENV = locate_r_runtime()


def run_figure_script(fig_num: int, force_rerun: bool = False):
    if fig_num not in range(1, 7):
        raise ValueError("Choose a figure number from 1 through 6")
    script_path = SCRIPT_DIR / f"plot_figure{fig_num}.R"
    data_dir = DATA_ROOT / f"figure{fig_num}"
    output_dir = OUTPUT_ROOT / f"figure{fig_num}"
    output_png = output_dir / f"Figure_{fig_num}_composite.png"
    output_pdf = output_dir / f"Figure_{fig_num}_composite.pdf"
    if not script_path.is_file():
        raise FileNotFoundError(f"Script not found: {script_path}")
    if not data_dir.is_dir():
        raise FileNotFoundError(f"Staged data directory not found: {data_dir}")
    if force_rerun or not all(p.is_file() and p.stat().st_size > 0 for p in (output_png, output_pdf)):
        print(f"Executing {script_path.name} with {R_EXEC}")
        subprocess.run(
            [str(R_EXEC), "--vanilla", "--slave", "-f", str(script_path),
             "--args", "--data-dir", str(data_dir), "--out-dir", str(output_dir)],
            cwd=ROOT, env=R_ENV, check=True, capture_output=True, text=True,
        )
        if not all(p.is_file() and p.stat().st_size > 0 for p in (output_png, output_pdf)):
            raise RuntimeError(f"Figure {fig_num} did not produce both non-empty composite files")
    else:
        print(f"Using existing Figure {fig_num} composite files")
    return output_png, output_pdf


print("Figure scripts:", SCRIPT_DIR)
print("Staged data:", DATA_ROOT)
print("R executable:", R_EXEC)


## Figure 1 — Discovery structure and cohort context

Displays the discovery workflow, the 2,000-gene expression heatmap across 586 discovery samples, the principal-state embedding, study composition, source classification correspondence, available clinical summaries, and stored survival curves. The principal-state subset contains 497 samples, with 89 intermediate profiles retained in the discovery context.


In [ ]:
png_path, pdf_path = run_figure_script(1, force_rerun=False)
print(f"Figure 1 PNG: {png_path}")
print(f"Figure 1 PDF: {pdf_path}")
display(Image(filename=str(png_path), width=700))


## Figure 2 — Bulk-associated biological programs

Displays the selected expression patterns and enrichment results, WGCNA evidence, transcription-factor programs, and selected TME contrasts. Values and eligible sample sets come from the staged figure sources.


In [ ]:
png_path, pdf_path = run_figure_script(2, force_rerun=False)
print(f"Figure 2 PNG: {png_path}")
print(f"Figure 2 PDF: {pdf_path}")
display(Image(filename=str(png_path), width=700))


## Figure 3 — Single-cell program localization

Displays the reviewed atlas subset, compartment-level program sources, patient-level comparisons, and patient-by-compartment program profiles. C1/C2/C3 refer to the originating bulk-associated programs. Source-dataset holds, coverage eligibility, unresolved identities, and the restriction on malignant identity remain part of the interpretation.


In [ ]:
png_path, pdf_path = run_figure_script(3, force_rerun=False)
print(f"Figure 3 PNG: {png_path}")
print(f"Figure 3 PDF: {pdf_path}")
display(Image(filename=str(png_path), width=700))


## Figure 4 — Assay-specific and matched multi-omic evidence

Displays mutation and protein evidence, selected cross-assay programs and pathways, matched assignment correspondence, and the staged synthesis. Each source retains its assay-specific denominator and qualification.


In [ ]:
png_path, pdf_path = run_figure_script(4, force_rerun=False)
print(f"Figure 4 PNG: {png_path}")
print(f"Figure 4 PDF: {pdf_path}")
display(Image(filename=str(png_path), width=700))


## Figure 5 — Classifier development and external projection

Displays the 114-gene consensus classifier workflow, selection and gene evidence, internal performance, external frozen-program projections, exploratory external survival associations, and compact-candidate comparisons. The fixed 17-gene development candidate is exploratory. External projections retain their source-cohort and label-availability limits.


In [ ]:
png_path, pdf_path = run_figure_script(5, force_rerun=False)
print(f"Figure 5 PNG: {png_path}")
print(f"Figure 5 PDF: {pdf_path}")
display(Image(filename=str(png_path), width=700))


## Figure 6 — Clinical associations and treatment hypotheses

Displays clinical prognostic information, observed duct-type comparisons, survival summaries, and the selected drug-evidence and uncertainty views. Observed pathology, model-derived quantities, and treatment hypotheses retain their separate source definitions and eligible denominators.


In [ ]:
png_path, pdf_path = run_figure_script(6, force_rerun=False)
print(f"Figure 6 PNG: {png_path}")
print(f"Figure 6 PDF: {pdf_path}")
display(Image(filename=str(png_path), width=700))


## Composite file summary

The following cell reports the presence and sizes of the resolved PDF and PNG composites. This records file availability. Scientific interpretation follows the source tables, protocols, and figure legends.


In [ ]:
summary_records = []
for i in range(1, 7):
    pdf = OUTPUT_ROOT / f"figure{i}/Figure_{i}_composite.pdf"
    png = OUTPUT_ROOT / f"figure{i}/Figure_{i}_composite.png"
    summary_records.append({
        "Figure": f"Figure {i}",
        "PDF_Exists": pdf.is_file(),
        "PDF_Size_KB": round(pdf.stat().st_size / 1024, 1) if pdf.is_file() else 0,
        "PNG_Exists": png.is_file(),
        "PNG_Size_KB": round(png.stat().st_size / 1024, 1) if png.is_file() else 0,
        "Script": str(SCRIPT_DIR / f"plot_figure{i}.R"),
        "Data_Dir": str(DATA_ROOT / f"figure{i}"),
    })
import json
print(json.dumps(summary_records, indent=2))
